# Lab 9: Comparative advantage in code
**MST 0441: Consumers, Trade and Business Strategy**
*Second in-person block, session 9. About 45 minutes.*

Problem A9: Norway and Portugal, salmon and textiles, $L = L^* = 1200$ hours,
and unit labor requirements

| | salmon | textiles |
|---|---|---|
| Norway   | $a_1 = 2$ | $a_2 = 4$ |
| Portugal | $a_1^* = 12$ | $a_2^* = 6$ |

Norway has the lower labor requirement in both goods. The lab computes why
it still does not undersell Portugal in both, then asks an assistant to argue
the opposite so that you can identify where the argument fails.

### How this lab works

1. Run `Runtime -> Run all` first. The notebook runs as it stands. Read the
   output, then return to the top.
2. Do the cells marked YOUR TURN. Each asks for a number, a line, or a short
   function. The cells are independent; a wrong answer in one does not affect
   the others.
3. Each YOUR TURN ends with a `check(...)` that reports whether your answer
   matches. Nothing raises an error.
4. The last section, *Work with your assistant*, calls your model from code
   through `ask_model()`. One-time setup: the key guide on Canvas. No key?
   Every prompt is a plain string you can copy into a chat window instead;
   paste the reply where marked. Test each reply in code before accepting it.

Nothing to install. `numpy`, `matplotlib` and `requests` are preinstalled in
Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction

def check(name, got, want, tol=1e-9):
    """Friendly checker: prints, never raises."""
    if got is None:
        print(f"  ..  {name}: not filled in yet")
        return False
    try:
        if isinstance(want, (list, tuple, np.ndarray)):
            good = np.allclose(np.asarray(got, dtype=float),
                               np.asarray(want, dtype=float), atol=tol)
        elif isinstance(want, str):
            good = str(got).strip().lower() == want.strip().lower()
        elif isinstance(want, bool):
            good = bool(got) is want
        else:
            good = abs(float(got) - float(want)) < tol
    except Exception as e:
        print(f"  XX  {name}: could not compare ({type(e).__name__}: {e})")
        return False
    print(f"  {'OK ' if good else 'XX '} {name} = {got}" + ("" if good else f"   (expected {want})"))
    return good

print("ready")

## 1. The two frontiers  (given: just run it)

In [ ]:
L, Ls = 1200, 1200
a1, a2   = 2.0, 4.0          # Norway:   salmon, textiles
a1s, a2s = 12.0, 6.0         # Portugal: salmon, textiles

def ppf(L, a1, a2, ax, label, color):
    x_max, y_max = L / a1, L / a2
    ax.plot([0, x_max], [y_max, 0], lw=2, color=color, label=label)
    return x_max, y_max

fig, ax = plt.subplots(figsize=(6, 4.4))
n = ppf(L,  a1,  a2,  ax, "Norway",   "#3b5bdb")
p = ppf(Ls, a1s, a2s, ax, "Portugal", "#c92a2a")
ax.set_xlabel("salmon"); ax.set_ylabel("textiles")
ax.legend(frameon=False); ax.grid(alpha=.3)
ax.set_title("Production possibility frontiers"); plt.show()

print(f"Norway   intercepts: salmon {n[0]:g}, textiles {n[1]:g}, slope {a1/a2:g}")
print(f"Portugal intercepts: salmon {p[0]:g}, textiles {p[1]:g}, slope {a1s/a2s:g}")

## 2. YOUR TURN: autarky prices and who has what advantage

The slope of the frontier *is* the autarky relative price of salmon,
$p^a = a_1/a_2$. Fill in the blanks.

In [ ]:
p_autarky_NOR = None      # <-- YOUR TURN
p_autarky_POR = None      # <-- YOUR TURN

check("Norway autarky price of salmon",   p_autarky_NOR, 0.5)
check("Portugal autarky price of salmon", p_autarky_POR, 2.0)


def advantages(a1, a2, a1s, a2s):
    """Return (absolute_salmon, absolute_textiles, comparative_home)."""
    abs_salmon   = None       # <-- YOUR TURN: "Norway" or "Portugal"
    abs_textiles = None       # <-- YOUR TURN
    comp_home    = None       # <-- YOUR TURN: which good does Norway export?
    return abs_salmon, abs_textiles, comp_home

s, t, c = advantages(a1, a2, a1s, a2s)
check("absolute advantage in salmon",   s, "Norway")
check("absolute advantage in textiles", t, "Norway")
check("Norway exports",                 c, "salmon")

Norway has the absolute advantage in both goods and still exports only one.
The next cells show why.

## 3. YOUR TURN: consumption outside your own frontier

At a world relative price $p$ strictly between the two autarky prices, each
country specializes completely and then trades along a line of slope $-p$
through its production point.

Fill in the specialization rule, then vary $p$ and note where both countries
gain.

In [ ]:
def consumption_frontier(p, L, a1, a2):
    """With relative price p (salmon in terms of textiles), specialize in the
    good with the lower opportunity cost, then trade along slope -p."""
    autarky = a1 / a2
    if p > autarky:
        x_prod, y_prod = L / a1, 0.0     # specialize in salmon
    else:
        x_prod, y_prod = 0.0, L / a2     # specialize in textiles
    # trading line through (x_prod, y_prod) with slope -p
    intercept_y = None                   # <-- YOUR TURN: value of the bundle in textiles
    return x_prod, y_prod, intercept_y

xp, yp, iy = consumption_frontier(1.0, L, a1, a2)
print("Norway at p = 1:", xp, yp, iy)
check("Norway trade-line textile intercept at p=1", iy, 600)

xp2, yp2, iy2 = consumption_frontier(1.0, Ls, a1s, a2s)
print("Portugal at p = 1:", xp2, yp2, iy2)
check("Portugal trade-line textile intercept at p=1", iy2, 200)

In [ ]:
# Both countries gain only when p lies strictly between the autarky prices.
fig, ax = plt.subplots(figsize=(6.4, 4.4))
ppf(L,  a1,  a2,  ax, "Norway PPF",   "#3b5bdb")
for p_world, style in [(1.0, "-"), (0.75, "--")]:
    xp, yp, iy = consumption_frontier(p_world, L, a1, a2)
    if iy is not None:
        ax.plot([0, iy / p_world], [iy, 0], style, color="green", lw=1.6,
                label=f"trade line, p = {p_world}")
ax.set_xlabel("salmon"); ax.set_ylabel("textiles")
ax.legend(frameon=False); ax.grid(alpha=.3)
ax.set_title("Trade lets Norway consume outside its own frontier"); plt.show()

print("gains-from-trade range for BOTH countries: "
      f"{min(a1/a2, a1s/a2s)} < p < {max(a1/a2, a1s/a2s)}")

## 4. YOUR TURN: the wages, and why Norway does not undersell everywhere

Normalize $p_2 = 1$ and set the world price $p_1 = 1$. A country that exports a
good must have unit cost equal to its price: $w\,a_i = p_i$.

In [ ]:
p1_w, p2_w = 1.0, 1.0

w_NOR = None      # <-- YOUR TURN: Norway exports salmon, so w = p1 / a1
w_POR = None      # <-- YOUR TURN: Portugal exports textiles, so w* = p2 / a2s

check("Norway wage",   w_NOR, 0.5)
check("Portugal wage", w_POR, 1/6)

if w_NOR and w_POR:
    print(f"wage ratio w/w* = {w_NOR/w_POR:g}")
    print(f"Norway's unit cost of TEXTILES   = w  * a2  = {w_NOR*a2:g}  vs price {p2_w:g}")
    print(f"Portugal's unit cost of SALMON   = w* * a1s = {w_POR*a1s:g}  vs price {p1_w:g}")
    print("\nBoth are ABOVE the price, so neither country enters the other's export "
          "industry.\nNorway is 3x as productive in textiles but pays 3x the wage, "
          "so the cost advantage cancels.")

---
## Your session 9 tutor

It carries the Ricardian model and this session's problem set, and it can
run your own trade calculations.

In [ ]:
import json

# Your model, as a function. (The TUTOR notebook has its own ask() that
# talks to the course tutor; this is a different function, and both can
# live in one notebook.) Works with a free Gemini key (Google AI Studio)
# Course route: an OpenRouter key running Mistral. One-time setup: key icon
# in the left sidebar -> add secret OPENROUTER_API_KEY, allow notebook access.
# (A free Gemini key under GOOGLE_API_KEY also works, as a fallback.)
# Never paste a key into a cell.

# ---- WHICH MODEL THIS STUDENT GETS --------------------------------------
# INSTRUCTOR: set EXPERIMENT below. "same" gives everyone MODEL_A.
# "split" sends a random half to MODEL_A and the other half to MODEL_B,
# assigned from the student id, so the same student always lands in the
# same arm however many times they re-run the notebook.
EXPERIMENT = "same"                                  # "same" or "split"
MODEL_A    = "mistralai/mistral-medium-3-5"          # strong at tool calling
MODEL_B    = "mistralai/mistral-small-2603"          # smaller, cheaper
GEMINI_MODEL = "gemini-3.6-flash"                    # fallback route only

ACTIVE_MODEL = MODEL_A

def assign_model(student_id=""):
    """Pick this student's model. Deterministic: same id -> same arm."""
    global ACTIVE_MODEL
    if EXPERIMENT == "split" and student_id.strip():
        import hashlib
        digest = hashlib.sha256(student_id.strip().lower().encode()).hexdigest()
        ACTIVE_MODEL = MODEL_A if int(digest, 16) % 2 == 0 else MODEL_B
    else:
        ACTIVE_MODEL = MODEL_A
    return ACTIVE_MODEL

def _get_secret(name):
    """Colab Secrets first; an environment variable as the fallback, so the
    instructor can test outside Colab. Never a value pasted into a cell."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    import os
    return os.environ.get(name)

def _credentials():
    for secret, base, model in [
        ("OPENROUTER_API_KEY",
         "https://openrouter.ai/api/v1",
         ACTIVE_MODEL),
        ("GOOGLE_API_KEY",
         "https://generativelanguage.googleapis.com/v1beta/openai",
         GEMINI_MODEL),
    ]:
        key = _get_secret(secret)
        if key:
            return base, key, model
    return None

def _chat(messages, tools=None):
    """One HTTP call to the model. Returns its reply message, or None."""
    creds = _credentials()
    if creds is None:
        print("(no API key found -- see the key guide on Canvas)")
        return None
    base, key, model = creds
    payload = {"model": model, "messages": messages}
    if tools:
        payload["tools"] = tools
        payload["tool_choice"] = "auto"
    try:
        import requests
        r = requests.post(base + "/chat/completions",
                          headers={"Authorization": "Bearer " + key},
                          json=payload, timeout=90)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]
    except Exception as e:
        print(f"(the call failed: {type(e).__name__}: {e})")
        return None


def ask_model(prompt, system=None):
    """One plain call: text in, text out. No tools, no loop."""
    messages = ([{"role": "system", "content": system}] if system else [])
    messages.append({"role": "user", "content": prompt})
    reply = _chat(messages)
    return None if reply is None else reply.get("content")


# ---- THE AGENT ----------------------------------------------------------
# A model on its own only writes text. An agent is a model plus TOOLS plus
# a LOOP. Read Conversation.say() once: it is the whole idea, in 20 lines.

class Conversation:
    """An agent you can talk to. It remembers the exchange, and it can call
    your Python functions in the middle of answering -- or ask you a
    question first and wait for your reply."""

    def __init__(self, tools=None, registry=None, system=None, show=True):
        self.tools, self.registry = tools, registry or {}
        self.show = show
        self.messages = [{"role": "system", "content": system}] if system else []

    def say(self, text, max_steps=6):
        """Say something to the agent. Returns its reply, or None with no key."""
        self.messages.append({"role": "user", "content": text})
        for _ in range(max_steps):
            reply = _chat(self.messages, self.tools)
            if reply is None:
                return None
            self.messages.append(reply)
            calls = reply.get("tool_calls")
            if not calls:                       # no tool wanted: it is answering
                return reply.get("content")
            for call in calls:                  # it asked; WE run the function
                name = call["function"]["name"]
                args = json.loads(call["function"]["arguments"] or "{}")
                result = self.registry[name](**args)
                if self.show:
                    print(f"   [agent ran {name}({args})]")
                self.messages.append({"role": "tool", "tool_call_id": call["id"],
                                      "content": json.dumps(result)})
        return "(gave up: too many steps)"


def run_agent(question, tools, registry, max_steps=6, show=True):
    """One-shot version: ask once, get the answer. A Conversation of length 1."""
    return Conversation(tools, registry, show=show).say(question, max_steps)


# ---- THIS SESSION'S TUTOR ----------------------------------------------
# The rules are the same in every lab. What changes is the MODEL the agent
# is teaching and the PROBLEM SET it can look up -- both handed in below.

TUTOR_RULES = """You are a teaching assistant in a first-year microeconomics
course. Rules you always follow:
1. Explain the METHOD before any numbers, in the order the course teaches it.
2. Never invent numbers. Get them by calling your tools.
3. If a parameter you need has not been given, ASK for it and stop. Do not
   assume a value.
4. When you report a result, say in plain words what it means economically,
   including what a high and a low value of the key parameter would imply.
5. Asked about a problem set question, call get_problem FIRST so you work from
   the real wording. Explain the method and the setup. Do NOT give the final
   numbers: leave those to the student. If they show you an answer, check it
   with your tools and say only whether it is right and which step failed.
6. Asked to test the student, ask ONE question at a time and wait. Say whether
   the reply is right before asking the next. Test understanding rather than
   recall: ask why something holds, or what would change if a parameter moved.
7. Be brief. Six sentences at most."""


def make_tutor(session, model_text, problems, tools, registry, show=True):
    """Build this session's tutor: shared rules, this session's knowledge."""
    def get_problem(problem_id):
        return problems.get(problem_id,
                            "no problem " + str(problem_id) + " in this session")

    reg = dict(registry)
    reg["get_problem"] = get_problem
    tls = list(tools) + [{
        "type": "function",
        "function": {
            "name": "get_problem",
            "description": ("Fetch the exact wording of a problem from THIS "
                            "session's problem set, so you work from the real "
                            "question rather than one you imagined. Valid ids: "
                            + ", ".join(sorted(problems)) + "."),
            "parameters": {"type": "object", "properties": {
                "problem_id": {"type": "string", "enum": sorted(problems)}},
                "required": ["problem_id"]}}}]

    system = (TUTOR_RULES + """

THE MODEL YOU ARE TEACHING IN THIS SESSION:
""" + model_text + """

The student is working through a lab on exactly this material, and has the
problem set open beside them.""")
    return Conversation(tls, reg, system=system, show=show)


print("ask_model() and run_agent() ready")

In [ ]:
#@title Session 9 knowledge (double-click if you want to read it)
SESSION9_MODEL = """The Ricardian model. Two countries, two goods, labor the
only factor. Unit labor requirements a_i at home and a_i* abroad. AUTARKY
PRICES equal opportunity costs: p = a_1/a_2 at home, p* = a_1*/a_2* abroad.
COMPARATIVE advantage is about these RATIOS, and it determines the pattern of
trade: home exports good 1 if a_1/a_2 < a_1*/a_2*. ABSOLUTE advantage is about
LEVELS, and it determines wages, not the pattern of trade. Both countries gain
from trade at any world price strictly between the two autarky prices. The
classic confusion is thinking a country productive in everything undersells
everywhere: it cannot, because its wage rises with its productivity. Unit cost
is w*a_i, so a country three times as productive with a wage three times as
high has the same unit cost. 'Low foreign wages reflect low foreign
productivity' is the same fact stated twice."""

PROBLEMS9 = {
    "A9.1": ("Norway and Portugal produce salmon (good 1) and textiles "
             "(good 2). Labor is the only factor and each has L = L* = 1200 "
             "hours. Unit labor requirements: Norway a_1 = 2, a_2 = 4; "
             "Portugal a_1* = 12, a_2* = 6. Parts cover the PPFs, autarky "
             "prices, the pattern of comparative advantage, the range of world "
             "prices at which both gain, and the equilibrium wage ratio."),
}

def trade_tool(a1, a2, a1_star, a2_star):
    """Autarky prices, comparative advantage and the gains-from-trade range."""
    p, p_star = a1 / a2, a1_star / a2_star
    lo, hi = min(p, p_star), max(p, p_star)
    return {"autarky_price_home": round(p, 4),
            "autarky_price_foreign": round(p_star, 4),
            "home_exports": "good 1" if p < p_star else "good 2",
            "gains_range_for_world_price": [round(lo, 4), round(hi, 4)],
            "note": "prices are opportunity costs; ratios decide trade, levels decide wages"}

TOOLS9 = [{"type": "function", "function": {
    "name": "trade_tool",
    "description": ("Ricardian comparative advantage from unit labor "
                    "requirements. Returns each country's autarky relative "
                    "price (good 1 in terms of good 2), which good home "
                    "exports, and the range of world prices at which BOTH "
                    "countries gain from trade."),
    "parameters": {"type": "object", "properties": {
        "a1": {"type": "number", "description": "home hours per unit of good 1"},
        "a2": {"type": "number", "description": "home hours per unit of good 2"},
        "a1_star": {"type": "number", "description": "foreign hours per unit of good 1"},
        "a2_star": {"type": "number", "description": "foreign hours per unit of good 2"}},
        "required": ["a1", "a2", "a1_star", "a2_star"]}}}]

tutor9 = make_tutor(9, SESSION9_MODEL, PROBLEMS9, TOOLS9, {"trade_tool": trade_tool})
print("session 9 tutor ready")

### Make it argue against you, then defend the model

First the adversary. Then hand the same agent the tools and let it settle
the argument with numbers.

In [ ]:
print(tutor9.say("A politician says Portugal is made WORSE OFF by trading "
                 "with Norway, because Norway is more productive in every "
                 "good. Use your tools on A9.1's numbers, then tell me exactly "
                 "which step of that argument is wrong.") or "(no key)")

In [ ]:
print(tutor9.say("Now ask me one question to check I understand the difference "
                 "between comparative and absolute advantage.") or "(no key)")

In [ ]:
my_reply = ""      # <-- YOUR TURN: answer in your own words
print(tutor9.say(my_reply) if my_reply else "answer the question above first")

> An assistant asked to be persuasive will be persuasive whether or not it is
> right. The defense is a computed number, which is why the wage ratio comes
> before the reading.

---
## Take away

* The pattern of trade follows **comparative** advantage (ratios $a_1/a_2$); the
  level of wages follows **absolute** advantage (levels $a_i$).
* Both countries gain only for $p$ strictly between the two autarky prices.
* Norway does not undersell everywhere because its wage rises with its
  productivity. "Low foreign wages reflect low foreign productivity" is the
  same statement.